# Multi-Qubit Gates & Entanglement

Create Bell states, explore CNOT, SWAP, and Toffoli gates.

**Objectives:**
- Build entangled states using CNOT and verify correlations
- Create all four Bell states
- Use SWAP and Toffoli (CCNOT) gates
- Understand how entanglement differs from classical correlation

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import matplotlib.pyplot as plt

from lib.circuits import bell_pair, ghz_state
from lib.utils.results import parse_counts

device = LocalSimulator()

## 1. CNOT Gate and the Bell Pair

CNOT (Controlled-NOT) flips the target qubit if and only if the control qubit is |1>.
Combined with Hadamard, it creates entanglement.

In [ ]:
# CNOT truth table: verify on all 4 basis states
print("CNOT Truth Table (control=q0, target=q1):")
print(f"{'Input':<10} {'Output':<10}")
print("-" * 20)

for input_state in ['00', '01', '10', '11']:
    circuit = Circuit()
    if input_state[0] == '1':
        circuit.x(0)
    if input_state[1] == '1':
        circuit.x(1)
    circuit.cnot(0, 1)
    
    result = device.run(circuit, shots=10).result()
    output = list(result.measurement_counts.keys())[0]
    print(f"|{input_state}>    ->  |{output}>")

In [ ]:
# Bell pair: H then CNOT creates (|00> + |11>) / sqrt(2)
circuit = bell_pair()
print(circuit)

result = device.run(circuit, shots=2000).result()
counts = parse_counts(result)

print(f"\nMeasurement counts: {dict(counts)}")
print("\nVerifying entanglement: every measurement shows both qubits agree.")

for bitstring in counts:
    assert bitstring[0] == bitstring[1], f"Correlation broken: {bitstring}"
# "They always agree" is NOT evidence of entanglement on its own: drop the Hadamard
# and the circuit collapses to the separable product state |00>, whose qubits also
# always agree, so the loop above passes vacuously. Entanglement needs BOTH branches
# to actually occur -- require them (each ~50% at 2000 shots).
assert set(counts) == {"00", "11"}, f"expected both |00> and |11>, got {dict(counts)}"
print("All measurements show perfect qubit correlation, and BOTH branches occur.")

## 2. All Four Bell States

There are four maximally entangled two-qubit states (Bell basis):
- |Phi+> = (|00> + |11>) / sqrt(2) -- H, CNOT
- |Phi-> = (|00> - |11>) / sqrt(2) -- Z, H, CNOT
- |Psi+> = (|01> + |10>) / sqrt(2) -- X on q1, H, CNOT
- |Psi-> = (|01> - |10>) / sqrt(2) -- X on q1, Z, H, CNOT

In [ ]:
# Create all four Bell states and compare their measurement statistics
bell_states = {
    "|Phi+>": Circuit().h(0).cnot(0, 1),
    "|Phi->": Circuit().h(0).z(0).cnot(0, 1),
    "|Psi+>": Circuit().h(0).cnot(0, 1).x(1),
    "|Psi->": Circuit().h(0).z(0).cnot(0, 1).x(1),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, (name, circuit) in zip(axes, bell_states.items()):
    result = device.run(circuit, shots=2000).result()
    counts = result.measurement_counts
    
    labels = sorted(counts.keys())
    values = [counts[label] / 2000 for label in labels]
    
    ax.bar(labels, values, color='#232f3e')
    ax.set_title(name)
    ax.set_ylim(0, 0.7)
    ax.set_ylabel('Probability')

plt.suptitle('The Four Bell States', fontsize=12)
plt.tight_layout()
plt.show()

print("|Phi+/-> produce |00> and |11> (qubits agree)")
print("|Psi+/-> produce |01> and |10> (qubits disagree)")

## 3. GHZ State (Multi-Qubit Entanglement)

The GHZ state generalizes the Bell pair to n qubits: (|00...0> + |11...1>) / sqrt(2).
Measuring any single qubit collapses all the others.

In [ ]:
# Create and verify GHZ states of increasing size
for n in [3, 4, 5]:
    circuit = ghz_state(n_qubits=n)
    result = device.run(circuit, shots=1000).result()
    counts = result.measurement_counts
    
    all_zeros = '0' * n
    all_ones = '1' * n
    
    print(f"{n}-qubit GHZ: {dict(counts)}")
    assert set(counts.keys()) == {all_zeros, all_ones}, f"Unexpected states in {n}-qubit GHZ!"

print("\nAll GHZ states produce only all-zeros or all-ones outcomes.")

## 4. SWAP Gate

The SWAP gate exchanges the states of two qubits. It can be decomposed into three CNOTs.

In [ ]:
# Prepare q0=|1>, q1=|0>, then SWAP
circuit = Circuit().x(0).swap(0, 1)
print(circuit)

result = device.run(circuit, shots=100).result()
counts = result.measurement_counts
print(f"\nAfter SWAP: {dict(counts)}")
print("q0 was |1>, q1 was |0> -> after SWAP: q0=|0>, q1=|1> -> outcome '01'")

# Verify: SWAP is equivalent to 3 CNOTs
circuit_3cnot = Circuit().x(0).cnot(0, 1).cnot(1, 0).cnot(0, 1)
result2 = device.run(circuit_3cnot, shots=100).result()
counts2 = result2.measurement_counts
print(f"3-CNOT decomposition: {dict(counts2)}")
assert counts == counts2, "SWAP and 3-CNOT should give identical results"
print("Confirmed: SWAP = CNOT * CNOT * CNOT")

## 5. Toffoli Gate (CCNOT)

The Toffoli gate has two controls and one target. It flips the target only when both controls are |1>.
This gate is universal for classical reversible computation.

In [ ]:
# Toffoli truth table
print("Toffoli Truth Table (controls=q0,q1, target=q2):")
print(f"{'Input':<10} {'Output':<10} {'Target flipped?'}")
print("-" * 35)

for q0 in [0, 1]:
    for q1 in [0, 1]:
        for q2 in [0, 1]:
            circuit = Circuit()
            if q0:
                circuit.x(0)
            if q1:
                circuit.x(1)
            if q2:
                circuit.x(2)
            circuit.ccnot(0, 1, 2)
            
            result = device.run(circuit, shots=10).result()
            output = list(result.measurement_counts.keys())[0]
            flipped = 'YES' if output[2] != str(q2) else 'no'
            print(f"|{q0}{q1}{q2}>    ->  |{output}>     {flipped}")

## 6. Exercises

Two exercises to lock in the multi-qubit toolkit. Each has tiered hints --
expand only what you need -- and a check cell that tells you when you have it.


### Exercise 1 — The |Phi-> Bell state

Section 2 built all four Bell states, but sampled counts alone cannot tell
|Phi-> from |Phi+> — the minus sign lives in a relative phase that measurement
statistics never see. Build |Phi-> = (|00> - |11>) / sqrt(2) yourself and
prove the sign is really there.

Define `phi_minus` — your circuit — and `phi_minus_counts` — the measurement
counts from running it with at least 1000 shots. The check inspects the counts
AND the exact amplitudes (via `statevector`), so |Phi+> will not pass.

<details><summary>Hint 1 — nudge</summary>

|Phi-> differs from the Section 1 Bell pair by a single relative sign. Which
single-qubit gate flips the sign of |1> while leaving |0> untouched? The
recipe list at the top of Section 2 shows where it slots in.

</details>
<details><summary>Hint 2 — approach</summary>

Take the Bell-pair chain from Section 1 and insert the phase-flip gate on
qubit 0 between the Hadamard and the CNOT. Run with
`device.run(..., shots=...)` and `parse_counts(...)` for the counts — the
check handles the amplitude inspection for you.

</details>


In [ ]:
# Exercise 1: Create the |Phi-> Bell state (|00> - |11>) / sqrt(2).
# Define: phi_minus -- the circuit -- and phi_minus_counts -- measurement
# counts from at least 1000 shots.

# TODO: your code here


In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check
from lib.utils.statevector import statevector

with check("Exercise 1"):
    assert sum(phi_minus_counts.values()) >= 1000, "run at least 1000 shots"
    assert set(phi_minus_counts) <= {"00", "11"}, (
        "|Phi-> should only ever measure |00> or |11>"
    )
    assert len(phi_minus_counts) == 2, "with 1000+ shots you should see BOTH outcomes"
    _sv = statevector(phi_minus)
    assert len(_sv) == 4, "phi_minus should act on exactly 2 qubits"
    assert abs(_sv[1]) < 1e-6 and abs(_sv[2]) < 1e-6, (
        "|01> and |10> should carry no amplitude"
    )
    assert abs(abs(_sv[0]) - 2**-0.5) < 1e-6 and abs(abs(_sv[3]) - 2**-0.5) < 1e-6, (
        "|00> and |11> should each have amplitude of magnitude 1/sqrt(2)"
    )
    assert (_sv[0] * _sv[3].conjugate()).real < -0.4, (
        "the |00> and |11> amplitudes must carry OPPOSITE signs -- that relative "
        "minus is what separates |Phi-> from |Phi+>"
    )


### Exercise 2 — A quantum AND gate

The Toffoli truth table in Section 5 flips the target exactly when both
controls are |1>. Read that as logic: a target initialized to |0> ends up
carrying `a AND b` — a reversible AND gate.

Define `and_results`: a dict mapping every input pair `(a, b)` — all four
combinations of 0 and 1 — to the measured target-qubit bit as an int.

<details><summary>Hint 1 — nudge</summary>

A target starting at |0> ends at |0 XOR (a AND b)> = |a AND b>. Nothing here
is ever in superposition, so every input combination gives one deterministic
outcome — a handful of shots per run is plenty.

</details>
<details><summary>Hint 2 — approach</summary>

Loop over the four `(a, b)` pairs. For each: build a fresh `Circuit()`, use
`x(...)` to raise whichever input qubits should start at |1>, apply
`ccnot(0, 1, 2)`, run it, and read the target's position in the outcome
bitstring into `and_results[(a, b)]`.

</details>


In [ ]:
# Exercise 2: Build a quantum AND gate from the Toffoli.
# Define: and_results -- dict mapping each input pair (a, b), for all four
# combinations, to the measured target bit as an int.

# TODO: your code here


In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert set(and_results) == {(0, 0), (0, 1), (1, 0), (1, 1)}, (
        "test all four (a, b) input combinations, keyed as tuples"
    )
    assert all(bit in (0, 1) for bit in and_results.values()), (
        "record each measured target bit as an int, 0 or 1"
    )
    assert all(and_results[(a, b)] == (a & b) for a, b in and_results), (
        "the target should read 1 exactly when both inputs are 1"
    )


## Summary

- **CNOT**: conditional flip, creates entanglement when control is in superposition
- **Bell states**: 4 maximally entangled 2-qubit states, foundation of quantum protocols
- **GHZ state**: n-qubit entanglement where all qubits are correlated
- **SWAP**: exchanges qubit states, decomposes into 3 CNOTs
- **Toffoli (CCNOT)**: 2-control gate, implements reversible classical logic
- CNOT + single-qubit gates form a universal gate set

**Next:** [04-measurement-statistics.ipynb](04-measurement-statistics.ipynb) -- understand shot-based measurement and statistical accuracy.